# TNBC miRNA–mRNA GAT study: phase 2 on Kaggle

Runs the phase-2 work that `study-final` (commit `59ca5e0`) skipped for compute budget, with the frozen
code, settings, seeds and phase-1 configuration selections. Nothing in `study/` is changed.

| Item | Fits |
|---|---|
| Learning curve, 25% and 50% of inner-fit groups: joint_mirdb, joint_nocross (repeat 0, 5 folds × 5 seeds) | 100 |
| Sensitivity: TCGA normal-derived edges (`sens_tcga_normal`) | 25 |
| Sensitivity: ESTIMATE-purity TCGA edges (`sens_tcga_estimate`) | 25 |
| Sensitivity: train-side random within-tissue pairing augmentation (`sens_aug_pairs`) | 25 |
| Learning-curve baselines (`study/baselines.py lc`: XGBoost joint/routed, elastic net routed) | 50 fold files |

**Settings:** Internet **on**. Accelerator **None (CPU)** is enough and recommended: the frozen model is
CPU-only and single-threaded per fit (`torch.set_num_threads(1)`), so a GPU is never used. GPU T4 x2 also
works and runs the same CPU code.

**Smoke test first:** run with `SMOKE_TEST = True` (3 real fits, about 5–10 min including installs). Then set
`SMOKE_TEST = False` and use *Save Version → Save & Run All*. Fits are checkpointed one CSV per fit in
`/kaggle/working/phase2_runs`; smoke-test fits are real fits and are reused.

**Resume:** finished fits are never refit. If a session ends early, attach this notebook's previous output
(*Add Input → Your Work → this notebook*) and run again: `restore` copies the finished fits back.
No new fit starts after `TIME_BUDGET_H − RESERVE_H` hours, so a run always finishes cleanly inside Kaggle's 12 h limit.


In [ ]:
import os, subprocess, time

SMOKE_TEST = True          # True: 3 real phase-2 fits to check the environment and timing. False: everything.
WORKERS = os.cpu_count()   # parallel fits; each fit is single-threaded (frozen setting)
TIME_BUDGET_H = 11.0       # Kaggle sessions stop at 12 h
RESERVE_H = 1.0            # kept for the baselines, analysis and zip; no new GAT fit starts after budget - reserve

BRANCH = 'kaggle-phase2'
REPO_URL = 'https://github.com/heramn/TNBC-miRNA-mRNA-GNN.git'
SRC = '/tmp/TNBC-miRNA-mRNA-GNN'
PY = '/tmp/p2env/bin/python'
WORK = '/kaggle/working'
T0 = time.time()
DEADLINE = T0 + (TIME_BUDGET_H - RESERVE_H) * 3600
os.environ.update(STUDY_RUNS=f'{WORK}/phase2_runs', PHASE2_OUT=f'{WORK}/phase2_output', STUDY_CACHE='/tmp/study_cache',
                  OMP_NUM_THREADS='1', OPENBLAS_NUM_THREADS='1', MKL_NUM_THREADS='1')


def sh(cmd, check=True):
    print('$', cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end='', flush=True)
    if p.wait() and check:
        raise RuntimeError(f'exit {p.returncode}: {cmd}')
    return p.returncode

print('workers:', WORKERS, '| smoke test:', SMOKE_TEST)

## 1. Clone the repository at `kaggle-phase2`

In [ ]:
sh(f'rm -rf {SRC} && git clone -q --depth 1 -b {BRANCH} {REPO_URL} {SRC} && git -C {SRC} log -1 --format="%h %s"')

## 2. Python environment

An isolated Python 3.12 venv with the exact versions phase 1 used (`requirements.txt`, REPRO_REPORT.md):
torch **2.15.0+cpu** and torch_geometric **2.8.1**. Using phase 1's CPU torch keeps the fits numerically
comparable; Kaggle's preinstalled CUDA torch would not be used by the CPU-only model anyway, and the
isolated venv keeps it from mixing with PyG. torch_geometric 2.8.1 is pure Python (GATConv/GCNConv need
no torch-scatter/torch-sparse wheels), so there is no CUDA-specific PyG wheel to match.
scikit-learn must stay at 1.7.x: 1.8 changes `StratifiedGroupKFold(shuffle=True)` splits.

In [ ]:
sh('pip install -q uv')
sh('uv venv -q --python 3.12 /tmp/p2env')
sh(f'uv pip install -q --python {PY} numpy==2.5.5 pandas==3.1.1 scipy==1.18.1 scikit-learn==1.7.2 statsmodels==0.15.0 '
   'matplotlib==3.11.2 xgboost==3.4.1 pydeseq2==0.6.0 anndata==0.13.4 torch_geometric==2.8.1 pytest')
sh(f'uv pip install -q --python {PY} torch==2.15.0 --index-url https://download.pytorch.org/whl/cpu')
sh(f'{PY} -c "import sys, torch, torch_geometric, sklearn, numpy; print(sys.version.split()[0], torch.__version__, '
   'torch_geometric.__version__, sklearn.__version__, numpy.__version__)"')

## 3. Data, checkpoint restore and frozen-file tests

* GEO GSE45498 inputs ship in the repository (`data/raw/`) and are checked by sha256 against the files phase 1 used.
* miRDB v6.0 is downloaded from mirdb.org and checked by sha256. If mirdb.org is unreachable, attach a Kaggle Dataset
  containing `miRDB_v6.0_prediction_result.txt.gz` (see `kaggle/README.md`); it is found anywhere under `/kaggle/input`.
* TCGA is not downloaded: the phase-2 arms use the frozen TCGA edge lists in `study/frozen/`.
* `pytest study/tests` includes the sha256 check of every frozen study file.

In [ ]:
os.chdir(SRC)
sh(f'{PY} kaggle/phase2.py data')
sh(f'{PY} kaggle/phase2.py restore')
sh(f'{PY} -m pytest -q -p no:cacheprovider study/tests')
sh(f'{PY} kaggle/phase2.py plan')

## 4. GAT fits

Each fit goes through the frozen `run_gat.run_one` with the arguments `run_gat.run_task` would pass, longest fits
first, one process per fit. Progress lines show elapsed seconds, status (`ran`, `cached`, `deferred` by the time
budget) and per-fit seconds; a per-fit log is written to `phase2_output/fit_log.csv`.

In [ ]:
smoke = '--smoke 3' if SMOKE_TEST else ''
sh(f'{PY} kaggle/phase2.py run {smoke} --workers {WORKERS} --deadline {DEADLINE:.0f}')

## 5. Learning-curve baselines, analysis and updated RESULTS.md rows

Full mode only. Runs `study/baselines.py lc`, then consolidates the phase-2 run files with `analysis.collect`,
appends them to the committed phase-1 `predictions_final.csv.gz`, reruns `analysis.analyze` and `figures.py` unchanged,
and prints the RESULTS.md rows for sections 9, 10 and 13. If any fit is still missing it says so and stops; rerun
the notebook with this output attached.

In [ ]:
if SMOKE_TEST:
    print('smoke test: skipping baselines and analysis')
else:
    sh(f'{PY} kaggle/phase2.py baselines --workers {WORKERS}')
    rc = sh(f'{PY} kaggle/phase2.py analyze', check=False)
    print('analysis', 'finished' if rc == 0 else f'not run (exit {rc}); attach this output and rerun to finish the missing fits')

## 6. Zip the outputs

`/kaggle/working/phase2_runs/` holds every per-run CSV (the checkpoint), `/kaggle/working/phase2_output/` the tables,
figures, consolidated predictions, fit log and `RESULTS_phase2_rows.md`; `tnbc_phase2.zip` holds both.

In [ ]:
import shutil, zipfile
os.chdir(WORK)
z = shutil.make_archive(f'{WORK}/tnbc_phase2', 'zip', WORK, '.')
with zipfile.ZipFile(z) as zf:
    names = [n for n in zf.namelist() if not n.endswith('/')]
names = [n for n in names if n.startswith(('phase2_runs/', 'phase2_output/'))]
print(z, f'{os.path.getsize(z) / 1e6:.1f} MB')
print(len([n for n in names if n.startswith('phase2_runs/') and n.endswith('.csv')]), 'per-run CSVs')
rows = f'{WORK}/phase2_output/RESULTS_phase2_rows.md'
if os.path.exists(rows):
    print(open(rows).read())
print(f'total time {(time.time() - T0) / 3600:.2f} h')